# BrainScanAI — Benchmark de 4 modèles pour la classification de tumeurs cérébrales

**Objectif :** comparer 4 technologies de vision sur la même tâche (glioma / meningioma /
notumor / pituitary) afin de montrer deux choses :

1. **Précision** : comment les architectures récentes améliorent le diagnostic.
2. **Coût / facilité** : entraînement (peu de données, peu de paramètres) et déploiement
   (taille, latence) — l'argument clé pour un usage clinique.

| Modèle | Techno | Année | Rôle dans l'histoire |
|---|---|---|---|
| `microsoft/resnet-50` | CNN supervisé | 2015 | Point de départ (référence) |
| `google/vit-base-patch16-224` | Vision Transformer | 2021 | Le saut "attention" |
| `facebook/dinov2-base` | Auto-supervisé (foundation) | 2023 | Entraînement léger, généralisation |
| `microsoft/BiomedCLIP-...` | Multimodal médical | 2023 | Frontière + zero-shot (0 entraînement) |

**Régimes évalués :** linear-probe (backbone gelé + tête) et fine-tuning complet pour les 3
modèles Hugging Face ; zero-shot et linear-probe pour BiomedCLIP.

> ⚠️ Outil de recherche / démonstration pédagogique, **pas un dispositif médical**.

## 0. Installation des dépendances supplémentaires
`open_clip_torch` (pour BiomedCLIP) et `imagehash` (déduplication du jeu externe).
Décommente si besoin.

In [ ]:
!pip install --quiet open_clip_torch imagehash scikit-learn pandas matplotlib

## 1. Imports, configuration et reproductibilité

In [ ]:
import os
import glob
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import AutoImageProcessor, AutoModelForImageClassification

warnings.filterwarnings("ignore")

# ----------------------------- CONFIG -----------------------------
SEED = 42
DATA_DIR = "./data"                  # contient Training/ et Testing/ (dataset Kaggle)
OOD_DIR = None                       # ex: "./data_external" pour le test de généralisation
                                     # (None => colonnes OOD laissées vides)
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = len(CLASS_NAMES)

BATCH_SIZE = 32
NUM_WORKERS = 4
VAL_FRACTION = 0.15                  # part de Training/ réservée à la validation

# Régimes : on garde des epochs modestes (modèles pré-entraînés) — ajustables.
EPOCHS = {"linear_probe": 5, "full_ft": 4}
LR = {"linear_probe": 1e-3, "full_ft": 3e-5}

QUICK_TEST = False                   # True => sous-échantillonne pour un dry-run rapide
QUICK_PER_CLASS = 60                 # images/classe en mode QUICK_TEST
# ------------------------------------------------------------------

# Mapping figé (ne dépend PAS de l'ordre de glob -> reproductible)
id2label = {i: c for i, c in enumerate(CLASS_NAMES)}
label2id = {c: i for i, c in id2label.items()}

# Alias de dossiers pour un éventuel jeu externe nommé différemment
# (édite selon la structure du dataset que tu télécharges).
CLASS_FOLDER_ALIASES = {
    "glioma": ["glioma", "Glioma", "glioma_tumor"],
    "meningioma": ["meningioma", "Meningioma", "meningioma_tumor"],
    "notumor": ["notumor", "no_tumor", "No_Tumor", "normal", "healthy"],
    "pituitary": ["pituitary", "Pituitary", "pituitary_tumor"],
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print(f"Device : {device} | AMP : {use_amp}")

## 2. Données : split propre train / val / test

- **train + val** proviennent du dossier `Training/` (split stratifié reproductible).
- **test** = dossier `Testing/` officiel (le vrai held-out, ignoré dans le projet initial).
- **OOD** (optionnel) = jeu externe pour mesurer la généralisation hors distribution.

In [ ]:
def list_split(root: str, aliases: dict | None = None) -> tuple[list, list]:
    """Liste (chemins, labels) en mappant chaque classe par nom de dossier (robuste)."""
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    files, labels = [], []
    for idx, cname in enumerate(CLASS_NAMES):
        candidates = aliases[cname] if aliases else [cname]
        folder = None
        for cand in candidates:
            p = os.path.join(root, cand)
            if os.path.isdir(p):
                folder = p
                break
        if folder is None:
            print(f"  ⚠️ dossier introuvable pour '{cname}' dans {root}")
            continue
        for p in sorted(glob.glob(os.path.join(folder, "*"))):
            if p.lower().endswith(exts):
                files.append(p)
                labels.append(idx)
    return files, labels


def subsample(files, labels, per_class):
    """Réduit à `per_class` images par classe (pour QUICK_TEST)."""
    by_class = {}
    for f, y in zip(files, labels):
        by_class.setdefault(y, []).append(f)
    nf, nl = [], []
    for y, fs in by_class.items():
        for f in fs[:per_class]:
            nf.append(f)
            nl.append(y)
    return nf, nl


train_root = os.path.join(DATA_DIR, "Training")
test_root = os.path.join(DATA_DIR, "Testing")

all_train_files, all_train_labels = list_split(train_root)
test_files, test_labels = list_split(test_root)

if QUICK_TEST:
    all_train_files, all_train_labels = subsample(all_train_files, all_train_labels, QUICK_PER_CLASS)
    test_files, test_labels = subsample(test_files, test_labels, QUICK_PER_CLASS // 3 + 1)

train_files, val_files, train_labels, val_labels = train_test_split(
    all_train_files, all_train_labels,
    test_size=VAL_FRACTION, stratify=all_train_labels, random_state=SEED,
)

print(f"Train : {len(train_files)} | Val : {len(val_files)} | Test (officiel) : {len(test_files)}")
print("Répartition test :", {CLASS_NAMES[i]: test_labels.count(i) for i in range(NUM_CLASSES)})

### 2bis. (Optionnel) Jeu externe + déduplication par hash perceptuel

Indispensable avant tout test de généralisation : on retire du jeu externe les images
qui seraient des doublons (quasi-)identiques de notre train, pour ne pas fausser la mesure.

In [ ]:
ood_files, ood_labels = [], []
if OOD_DIR is not None:
    ood_files, ood_labels = list_split(OOD_DIR, aliases=CLASS_FOLDER_ALIASES)
    if QUICK_TEST:
        ood_files, ood_labels = subsample(ood_files, ood_labels, QUICK_PER_CLASS // 3 + 1)
    print(f"OOD brut : {len(ood_files)} images")

    try:
        import imagehash

        def phash(path):
            return imagehash.phash(Image.open(path).convert("L"))

        print("Calcul des hash du train (déduplication)...")
        train_hashes = {phash(p) for p in train_files}

        kept_f, kept_l, dropped = [], [], 0
        for f, y in zip(ood_files, ood_labels):
            h = phash(f)
            if any((h - th) <= 5 for th in train_hashes):   # distance de Hamming <= 5
                dropped += 1
            else:
                kept_f.append(f)
                kept_l.append(y)
        ood_files, ood_labels = kept_f, kept_l
        print(f"OOD après dédup : {len(ood_files)} images ({dropped} doublons retirés)")
    except ImportError:
        print("imagehash non installé -> déduplication ignorée (à installer pour une mesure propre).")
else:
    print("Pas de jeu externe (OOD_DIR=None) : colonnes OOD laissées vides.")

## 3. Dataset et utilitaires communs

In [ ]:
class MRIDataset(Dataset):
    """Renvoie (image PIL RGB, label). Le prétraitement est fait dans la collate (HF)
    ou via un transform torchvision (BiomedCLIP)."""

    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[i]


def make_hf_collate(processor):
    """Applique le processor HF (resize + normalisation propres au modèle) sur le batch."""
    def collate(batch):
        imgs, labels = zip(*batch)
        enc = processor(list(imgs), return_tensors="pt")
        return enc["pixel_values"], torch.tensor(labels, dtype=torch.long)
    return collate


def set_regime(model, regime: str):
    """linear_probe : on ne réentraîne que la tête .classifier. full_ft : tout."""
    if regime == "linear_probe":
        for p in model.parameters():
            p.requires_grad = False
        head = getattr(model, "classifier", None)
        if head is None:
            raise ValueError("Pas d'attribut .classifier — adapter pour ce modèle.")
        for p in head.parameters():
            p.requires_grad = True
    else:  # full_ft
        for p in model.parameters():
            p.requires_grad = True


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def size_mb(total_params):
    return total_params * 4 / 1e6   # approximation fp32


@torch.no_grad()
def measure_latency_hf(model, sample_pixel_values, n=50):
    """Latence d'inférence (ms/image) sur 1 image, après warmup."""
    model.eval()
    x = sample_pixel_values[:1].to(device)
    for _ in range(5):
        model(pixel_values=x)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        model(pixel_values=x)
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000.0

## 4. Boucle d'entraînement et d'évaluation (modèles Hugging Face)

In [ ]:
def train_supervised(model, train_loader, val_loader, epochs, lr):
    """Entraîne le modèle (paramètres requires_grad=True uniquement). Renvoie le temps (s)."""
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss()
    dev_type = "cuda" if device == "cuda" else "cpu"

    t0 = time.time()
    for ep in range(epochs):
        model.train()
        running = 0.0
        for pixel_values, labels in train_loader:
            pixel_values = pixel_values.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.autocast(device_type=dev_type, enabled=use_amp):
                logits = model(pixel_values=pixel_values).logits
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * labels.size(0)
        val_acc, val_f1, _, _ = evaluate(model, val_loader)
        print(f"    epoch {ep + 1}/{epochs} | loss {running / len(train_loader.dataset):.4f} "
              f"| val_acc {val_acc:.4f} | val_f1 {val_f1:.4f}")
    return time.time() - t0


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, gts = [], []
    dev_type = "cuda" if device == "cuda" else "cpu"
    for pixel_values, labels in loader:
        pixel_values = pixel_values.to(device, non_blocking=True)
        with torch.autocast(device_type=dev_type, enabled=use_amp):
            logits = model(pixel_values=pixel_values).logits
        preds.append(logits.argmax(-1).cpu())
        gts.append(labels)
    preds = torch.cat(preds).numpy()
    gts = torch.cat(gts).numpy()
    acc = accuracy_score(gts, preds)
    f1 = f1_score(gts, preds, average="macro")
    return acc, f1, preds, gts

## 5. Benchmark des 3 modèles Hugging Face (ResNet-50, ViT-base, DINOv2-base)

Pour chaque modèle on recharge des poids frais à chaque régime (pas de fuite entre régimes).

In [ ]:
HF_MODELS = {
    "ResNet-50": "microsoft/resnet-50",
    "ViT-base": "google/vit-base-patch16-224",
    "DINOv2-base": "facebook/dinov2-base",
}

results = []          # liste de dicts -> DataFrame final
confusions = {}       # matrices de confusion (test interne) pour la viz

for model_name, model_id in HF_MODELS.items():
    print(f"\n{'=' * 70}\n {model_name}  ({model_id})\n{'=' * 70}")

    processor = AutoImageProcessor.from_pretrained(model_id)
    collate = make_hf_collate(processor)

    train_loader = DataLoader(MRIDataset(train_files, train_labels), batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate)
    val_loader = DataLoader(MRIDataset(val_files, val_labels), batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
    test_loader = DataLoader(MRIDataset(test_files, test_labels), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)
    ood_loader = None
    if ood_files:
        ood_loader = DataLoader(MRIDataset(ood_files, ood_labels), batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate)

    sample_pixel_values = next(iter(test_loader))[0]

    for regime in ["linear_probe", "full_ft"]:
        print(f"\n  --- Régime : {regime} ---")
        set_seed(SEED)
        model = AutoModelForImageClassification.from_pretrained(
            model_id, num_labels=NUM_CLASSES,
            id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True,
        )
        set_regime(model, regime)
        total, trainable = count_params(model)

        train_time = train_supervised(model, train_loader, val_loader,
                                       epochs=EPOCHS[regime], lr=LR[regime])

        acc_in, f1_in, preds_in, gts_in = evaluate(model, test_loader)
        print(f"    >> Test interne : acc {acc_in:.4f} | macro-F1 {f1_in:.4f}")
        if regime == "full_ft":
            print(classification_report(gts_in, preds_in, target_names=CLASS_NAMES, digits=3))
            confusions[model_name] = confusion_matrix(gts_in, preds_in)

        acc_ood = f1_ood = np.nan
        if ood_loader is not None:
            acc_ood, f1_ood, _, _ = evaluate(model, ood_loader)
            print(f"    >> Test externe (OOD) : acc {acc_ood:.4f} | macro-F1 {f1_ood:.4f}")

        latency = measure_latency_hf(model, sample_pixel_values)

        results.append({
            "model": model_name, "regime": regime,
            "trainable_params": trainable, "total_params": total,
            "size_mb": round(size_mb(total), 1),
            "train_time_s": round(train_time, 1), "latency_ms": round(latency, 2),
            "acc_internal": round(acc_in, 4), "f1_internal": round(f1_in, 4),
            "acc_ood": round(acc_ood, 4) if not np.isnan(acc_ood) else np.nan,
            "f1_ood": round(f1_ood, 4) if not np.isnan(f1_ood) else np.nan,
        })

        del model
        if device == "cuda":
            torch.cuda.empty_cache()

## 6. BiomedCLIP — zero-shot puis linear-probe

BiomedCLIP est distribué via `open_clip`. On l'utilise de deux façons :
- **zero-shot** : aucune image d'entraînement, juste des descriptions textuelles des classes.
- **linear-probe** : on extrait une fois les features (encodeur gelé) puis on entraîne un
  classifieur linéaire — illustration directe d'un entraînement quasi-gratuit.

In [ ]:
import open_clip

CLIP_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
clip_model, clip_preprocess = open_clip.create_model_from_pretrained(CLIP_ID)
clip_tokenizer = open_clip.get_tokenizer(CLIP_ID)
clip_model = clip_model.to(device).eval()

clip_total = sum(p.numel() for p in clip_model.parameters())

# DataLoaders dédiés (transform = preprocess de BiomedCLIP)
clip_train_loader = DataLoader(MRIDataset(train_files, train_labels, transform=clip_preprocess),
                               batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
clip_test_loader = DataLoader(MRIDataset(test_files, test_labels, transform=clip_preprocess),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
clip_ood_loader = None
if ood_files:
    clip_ood_loader = DataLoader(MRIDataset(ood_files, ood_labels, transform=clip_preprocess),
                                 batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# --- Prompts textuels (le "prompt engineering" fait partie de l'analyse) ---
PROMPTS = {
    "glioma": "an MRI scan of a brain with a glioma tumor",
    "meningioma": "an MRI scan of a brain with a meningioma tumor",
    "notumor": "an MRI scan of a healthy brain with no tumor",
    "pituitary": "an MRI scan of a brain with a pituitary tumor",
}
text_tokens = clip_tokenizer([PROMPTS[c] for c in CLASS_NAMES]).to(device)
with torch.no_grad():
    text_features = F.normalize(clip_model.encode_text(text_tokens), dim=-1)


@torch.no_grad()
def clip_extract_features(loader):
    """Renvoie (features L2-normalisées, labels)."""
    feats, ys = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        f = clip_model.encode_image(imgs)
        feats.append(F.normalize(f, dim=-1).cpu())
        ys.append(labels)
    return torch.cat(feats), torch.cat(ys)


@torch.no_grad()
def clip_zero_shot(loader):
    preds, gts = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        feats = F.normalize(clip_model.encode_image(imgs), dim=-1)
        logits = feats @ text_features.T
        preds.append(logits.argmax(-1).cpu())
        gts.append(labels)
    preds = torch.cat(preds).numpy()
    gts = torch.cat(gts).numpy()
    return accuracy_score(gts, preds), f1_score(gts, preds, average="macro"), preds, gts


@torch.no_grad()
def clip_latency(n=50):
    dummy = torch.randn(1, 3, 224, 224, device=device)
    for _ in range(5):
        clip_model.encode_image(dummy)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        clip_model.encode_image(dummy)
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000.0

### 6a. Zero-shot

In [ ]:
print("BiomedCLIP — zero-shot")
acc_zs, f1_zs, preds_zs, gts_zs = clip_zero_shot(clip_test_loader)
print(f"  >> Test interne : acc {acc_zs:.4f} | macro-F1 {f1_zs:.4f}")
print(classification_report(gts_zs, preds_zs, target_names=CLASS_NAMES, digits=3))
confusions["BiomedCLIP (zero-shot)"] = confusion_matrix(gts_zs, preds_zs)

acc_zs_ood = f1_zs_ood = np.nan
if clip_ood_loader is not None:
    acc_zs_ood, f1_zs_ood, _, _ = clip_zero_shot(clip_ood_loader)
    print(f"  >> Test externe (OOD) : acc {acc_zs_ood:.4f} | macro-F1 {f1_zs_ood:.4f}")

lat_clip = clip_latency()
results.append({
    "model": "BiomedCLIP", "regime": "zero_shot",
    "trainable_params": 0, "total_params": clip_total,
    "size_mb": round(size_mb(clip_total), 1),
    "train_time_s": 0.0, "latency_ms": round(lat_clip, 2),
    "acc_internal": round(acc_zs, 4), "f1_internal": round(f1_zs, 4),
    "acc_ood": round(acc_zs_ood, 4) if not np.isnan(acc_zs_ood) else np.nan,
    "f1_ood": round(f1_zs_ood, 4) if not np.isnan(f1_zs_ood) else np.nan,
})

### 6b. Linear-probe sur features gelées (entraînement quasi-instantané)

In [ ]:
print("BiomedCLIP — extraction des features (encodeur gelé)...")
t0 = time.time()
Xtr, ytr = clip_extract_features(clip_train_loader)
Xte, yte = clip_extract_features(clip_test_loader)
feat_dim = Xtr.shape[1]

# Petit classifieur linéaire entraîné sur les features mises en cache
set_seed(SEED)
linear = nn.Linear(feat_dim, NUM_CLASSES).to(device)
opt = torch.optim.AdamW(linear.parameters(), lr=1e-3, weight_decay=0.01)
crit = nn.CrossEntropyLoss()
Xtr_d, ytr_d = Xtr.to(device), ytr.to(device)

linear.train()
for ep in range(60):
    opt.zero_grad()
    loss = crit(linear(Xtr_d), ytr_d)
    loss.backward()
    opt.step()
train_time_lp = time.time() - t0   # inclut l'extraction des features

linear.eval()
with torch.no_grad():
    preds_lp = linear(Xte.to(device)).argmax(-1).cpu().numpy()
gts_lp = yte.numpy()
acc_lp = accuracy_score(gts_lp, preds_lp)
f1_lp = f1_score(gts_lp, preds_lp, average="macro")
print(f"  >> Test interne : acc {acc_lp:.4f} | macro-F1 {f1_lp:.4f}")
print(classification_report(gts_lp, preds_lp, target_names=CLASS_NAMES, digits=3))
confusions["BiomedCLIP (linear-probe)"] = confusion_matrix(gts_lp, preds_lp)

acc_lp_ood = f1_lp_ood = np.nan
if clip_ood_loader is not None:
    Xood, yood = clip_extract_features(clip_ood_loader)
    with torch.no_grad():
        preds_lp_ood = linear(Xood.to(device)).argmax(-1).cpu().numpy()
    acc_lp_ood = accuracy_score(yood.numpy(), preds_lp_ood)
    f1_lp_ood = f1_score(yood.numpy(), preds_lp_ood, average="macro")
    print(f"  >> Test externe (OOD) : acc {acc_lp_ood:.4f} | macro-F1 {f1_lp_ood:.4f}")

results.append({
    "model": "BiomedCLIP", "regime": "linear_probe",
    "trainable_params": sum(p.numel() for p in linear.parameters()),
    "total_params": clip_total,
    "size_mb": round(size_mb(clip_total), 1),
    "train_time_s": round(train_time_lp, 1), "latency_ms": round(lat_clip, 2),
    "acc_internal": round(acc_lp, 4), "f1_internal": round(f1_lp, 4),
    "acc_ood": round(acc_lp_ood, 4) if not np.isnan(acc_lp_ood) else np.nan,
    "f1_ood": round(f1_lp_ood, 4) if not np.isnan(f1_lp_ood) else np.nan,
})

## 7. Tableau récapitulatif et visualisations

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values("acc_internal", ascending=False).reset_index(drop=True)
df.to_csv("benchmark_results.csv", index=False)
print("Résultats sauvegardés -> benchmark_results.csv")
df

### 7a. Le graphe clé : précision vs nombre de paramètres entraînés
(l'argument "facilité d'entraînement" : monter à gauche = beaucoup de précision pour peu d'effort)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for _, r in df.iterrows():
    x = max(r["trainable_params"], 1)
    ax.scatter(x, r["acc_internal"], s=120)
    ax.annotate(f'{r["model"]}\n({r["regime"]})', (x, r["acc_internal"]),
                textcoords="offset points", xytext=(8, 4), fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Paramètres entraînés (échelle log)")
ax.set_ylabel("Accuracy (test interne)")
ax.set_title("Précision vs coût d'entraînement")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7b. Précision interne vs externe (robustesse / généralisation)

In [ ]:
labels_x = [f'{r["model"]}\n{r["regime"]}' for _, r in df.iterrows()]
x = np.arange(len(df))
w = 0.38
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - w / 2, df["acc_internal"], w, label="Test interne")
if df["acc_ood"].notna().any():
    ax.bar(x + w / 2, df["acc_ood"].fillna(0), w, label="Test externe (OOD)")
ax.set_xticks(x)
ax.set_xticklabels(labels_x, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Interne vs externe — la chute mesure la vraie généralisation")
ax.legend()
plt.tight_layout()
plt.show()

### 7c. Temps d'entraînement par configuration

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, df["train_time_s"])
ax.set_ylabel("Temps d'entraînement (s)")
ax.set_xticks(x)
ax.set_xticklabels(labels_x, rotation=45, ha="right", fontsize=8)
ax.set_title("Coût d'entraînement")
plt.tight_layout()
plt.show()

### 7d. Matrices de confusion (test interne)

In [ ]:
n = len(confusions)
if n:
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, cm) in zip(axes, confusions.items()):
        ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
            ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
        ax.set_title(name, fontsize=10)
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Comment interpréter / rédiger

Grille de lecture pour le rapport :

- **Précision (axe 1)** : ResNet sert de plancher. ViT-base devrait faire mieux après
  fine-tuning ; DINOv2 et BiomedCLIP montrent ce qu'apportent les features modernes.
- **Facilité d'entraînement (axe 2)** : compare `trainable_params` et `train_time_s`. Le
  linear-probe (et surtout BiomedCLIP sur features cachées) atteint une bonne précision
  en n'entraînant qu'une poignée de paramètres — c'est l'argument "hôpital sans gros GPU".
- **Déploiement** : `size_mb` + `latency_ms`. Mentionne qu'on peut encore réduire via
  quantification / export ONNX, et choisir des variantes `small` (ex. `dinov2-small`).
- **Généralisation (le plus convaincant)** : l'écart interne → externe. Un modèle qui
  chute peu hors distribution est bien plus utile en clinique réelle qu'un 98 % interne.
- **Zero-shot** : si BiomedCLIP est décevant en zero-shot, c'est un *résultat* à analyser
  (le pré-entraînement médical n'aide pas automatiquement sur une tâche aussi spécifique),
  et ça motive le fine-tuning.